In [ ]:
# ============================================================
# Original Paper Depth Recovery Test
# Strictly follows the described sequence in Pouli et al. (2010):
# luminance -> sigmoidal space -> bilateral filter
# -> inverse sigmoid -> optional reshaping -> depth map
# ============================================================

import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# ------------------------------------------------------------
# 1. Upload input image
# ------------------------------------------------------------

uploaded = files.upload()
image_path = list(uploaded.keys())[0]

image_bgr = cv2.imread(image_path)

if image_bgr is None:
    raise ValueError("Image loading failed. Please check the uploaded file.")

image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
image_float = image_rgb.astype(np.float32) / 255.0

height, width = image_float.shape[:2]

print("Image loaded:", image_path)
print("Image size:", width, "x", height)

# ------------------------------------------------------------
# 2. Convert RGB image to luminance
# ------------------------------------------------------------

R = image_float[:, :, 0]
G = image_float[:, :, 1]
B = image_float[:, :, 2]

luminance = 0.299 * R + 0.587 * G + 0.114 * B
luminance = np.clip(luminance, 0.0, 1.0)

# ------------------------------------------------------------
# 3. Convert luminance into sigmoidal space
# ------------------------------------------------------------
# The paper does not give the exact sigmoid formula/parameter.
# This is a standard logistic remapping used to implement the described step.

alpha = 10.0

sigmoid_luminance = 1.0 / (1.0 + np.exp(-alpha * (luminance - 0.5)))
sigmoid_luminance = np.clip(sigmoid_luminance, 1e-6, 1.0 - 1e-6)

# ------------------------------------------------------------
# 4. Apply bilateral filter in sigmoidal space
# ------------------------------------------------------------
# Table 2 "Same" settings:
# sigma_s = 1.2% of image width
# sigma_i = 16.8% of intensity range

sigma_s_percent = 1.2
sigma_i_percent = 16.8

sigma_space = width * (sigma_s_percent / 100.0)
sigma_color = sigma_i_percent / 100.0

sigmoid_filtered = cv2.bilateralFilter(
    sigmoid_luminance.astype(np.float32),
    d=0,
    sigmaColor=sigma_color,
    sigmaSpace=sigma_space
)

sigmoid_filtered = np.clip(sigmoid_filtered, 1e-6, 1.0 - 1e-6)

# ------------------------------------------------------------
# 5. Invert sigmoid
# ------------------------------------------------------------
# This is mathematical inverse sigmoid / logit,
# NOT image inversion.
#
# If y = 1 / (1 + exp(-alpha * (x - 0.5))),
# then x = log(y / (1 - y)) / alpha + 0.5

inverse_sigmoid = np.log(sigmoid_filtered / (1.0 - sigmoid_filtered)) / alpha + 0.5
inverse_sigmoid = np.clip(inverse_sigmoid, 0.0, 1.0)

# ------------------------------------------------------------
# 6. Optional reshaping
# ------------------------------------------------------------
# The paper mentions optional reshaping but does not define the exact function.
# Here we only apply the depth multiplier from Table 2 "Same": 0.35.
# No 1 - inverse operation is used.

depth_multiplier = 0.35

depth_map = inverse_sigmoid * depth_multiplier

# Normalize for saving and visualization
depth_norm = (depth_map - depth_map.min()) / (depth_map.max() - depth_map.min() + 1e-6)
depth_norm = np.clip(depth_norm, 0.0, 1.0)

print("Depth map generated.")
print("Depth range:", depth_norm.min(), "-", depth_norm.max())

# ------------------------------------------------------------
# 7. Save output images
# ------------------------------------------------------------

depth_gray = (depth_norm * 255).astype(np.uint8)

depth_gray_path = "paper_sequence_depth_gray.png"
cv2.imwrite(depth_gray_path, depth_gray)

depth_colormap = cv2.applyColorMap(depth_gray, cv2.COLORMAP_INFERNO)
depth_colormap_path = "paper_sequence_depth_colormap.png"
cv2.imwrite(depth_colormap_path, depth_colormap)

plt.figure(figsize=(16, 4))

plt.subplot(1, 4, 1)
plt.imshow(image_rgb)
plt.title("Input Image")
plt.axis("off")

plt.subplot(1, 4, 2)
plt.imshow(luminance, cmap="gray")
plt.title("Luminance")
plt.axis("off")

plt.subplot(1, 4, 3)
plt.imshow(sigmoid_filtered, cmap="gray")
plt.title("Filtered Sigmoid Space")
plt.axis("off")

plt.subplot(1, 4, 4)
plt.imshow(depth_norm, cmap="gray")
plt.title("Recovered Depth Map")
plt.axis("off")

plt.tight_layout()

comparison_path = "paper_sequence_depth_process.png"
plt.savefig(comparison_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved files:")
print(depth_gray_path)
print(depth_colormap_path)
print(comparison_path)

# ------------------------------------------------------------
# 8. Download outputs
# ------------------------------------------------------------

files.download(depth_gray_path)
files.download(depth_colormap_path)
files.download(comparison_path)